In [2]:
import sys
from pathlib import Path

In [3]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))
    print("Device Count:", torch.cuda.device_count())

CUDA available: False


In [4]:
PROJECT_ROOT = Path().resolve().parent.parent.parent

In [5]:
# Add project root to Python path
sys.path.append(str(PROJECT_ROOT))

In [6]:
import pandas as pd
from src.pipeline.utils.loader import load_jsonl
from src.pipeline.utils.document import Document
from collections import Counter
from typing import Any, Dict, Iterator
import plotly.express as px
import json
import numpy as np


In [7]:
DATA_PATH = PROJECT_ROOT / "kuatia" / "1_1_0" / "3_pii_censoring" / "censored"

In [8]:
OUTPUT_PATH = PROJECT_ROOT / "outputs" / "heuristics_exploration"

In [9]:
def iterate_directory(directory:Path) -> Iterator[Document]:
    for file in directory.glob('*.jsonl'):
        documents = load_jsonl(str(file), field_map={"id":"id", "text":"text"}, load_metadata=True, metadata_fields="*")
        for doc in documents:
            yield doc

In [10]:
documents = iterate_directory(DATA_PATH)

In [11]:
rows_all = []
for document in documents:
    row = {
        "id": document.id,
        "text": document.text
    }
    for field, value in document.metadata.items():
        if type(value) != "dict":
            row[field] = value

    #row["remove"] = document.id in affected_docs

    rows_all.append(row)
df_all = pd.DataFrame(rows_all)

In [ ]:
def compute_weighted_rel_nll(df, ppl_high_col, ppl_low_col, tokens_col):
        if len(df) == 0:
            return np.nan
        nll_high = np.log(df[ppl_high_col])
        nll_low = np.log(df[ppl_low_col])
        rel_nll = nll_high - nll_low

        total_tokens = df[tokens_col].sum()
        if total_tokens == 0:
            return np.nan
        return (df[tokens_col] * rel_nll).sum() / total_tokens

In [ ]:
def evaluate_filter_stage(
    df_base: pd.DataFrame,
    df_prev: pd.DataFrame,
    df_current: pd.DataFrame,
    ppl_high_col: str = "coreguapa_perplexity",
    ppl_low_col: str = "tweets_perplexity",
    tokens_col: str = "num_words_split",
    stage_name: str = "Filtered Stage",
) -> pd.Series:
    """Evaluates quality metrics comparing the current stage against both the original baseline

    and the immediately preceding filter stage.
    """

    # --- Token Yields ---
    base_tokens = df_base[tokens_col].sum()
    prev_tokens = df_prev[tokens_col].sum()
    curr_tokens = df_current[tokens_col].sum()

    cum_token_yield_pct = (
        (curr_tokens / base_tokens) * 100 if base_tokens > 0 else 0
    )
    step_token_loss_pct = (
        ((prev_tokens - curr_tokens) / prev_tokens) * 100
        if prev_tokens > 0
        else 0
    )

    # --- Relative NLL Calculations (Lower is Better) ---
    base_nll = compute_weighted_rel_nll(df_base)
    prev_nll = compute_weighted_rel_nll(df_prev)
    curr_nll = compute_weighted_rel_nll(df_current)

    # NLL Improvement = Old NLL - New NLL (Positive value means quality improved)
    step_nll_gain = prev_nll - curr_nll
    cum_nll_gain = base_nll - curr_nll

    # --- Efficiency (Quality Gain per % Tokens Dropped in this step) ---
    step_efficiency = (
        (step_nll_gain / step_token_loss_pct) if step_token_loss_pct > 0 else 0.0
    )

    return pd.Series(
        {
            "Stage": stage_name,
            "Docs Kept": len(df_current),
            "Cum. Token Yield (%)": round(cum_token_yield_pct, 2),
            "Step Tokens Dropped (%)": round(step_token_loss_pct, 2),
            "Weighted Rel NLL": round(curr_nll, 4),
            "Step Gain (vs Prev)": round(step_nll_gain, 4),
            "Cum Gain (vs Base)": round(cum_nll_gain, 4),
            "Step Efficiency": round(step_efficiency, 5),
        }
    )

In [61]:
results = []

In [62]:
df_baseline = df_all.copy()
df_prev = df_all.copy()

In [63]:
# --- Step 0: Baseline Entry ---
results.append(
    evaluate_filter_stage(
        df_base=df_baseline,
        df_prev=df_baseline,
        df_current=df_baseline,
        stage_name="0. Baseline (Unfiltered)",
    )
)

In [64]:
df_current = df_prev[(~(df_prev['count_sentences_with_legal_phrases'] > 0))]

In [67]:
results.append(evaluate_filter_stage(
    df_base=df_baseline,
    df_prev=df_prev,
    df_current = df_current,
    stage_name = "1. Having sentences with legal phrases"
))

results[-1]

Stage                      1. Having sentences with legal phrases
Docs Kept                                                  959074
Cum. Token Yield (%)                                         99.3
Step Tokens Dropped (%)                                       0.7
Weighted Rel NLL                                          -1.1523
Step Gain (vs Prev)                                         0.006
Cum Gain (vs Base)                                          0.006
Step Efficiency                                           0.00861
dtype: object

In [71]:
df_prev = df_current.copy()

df_current = df_current[~((df_current['count_sentences_with_curly_bracket'] > 0) & (df_current['corpus'] == 'opus-all-en') & (df_current['language_score'] < 0.9965))]

df_current = df_current[~((df_current['count_sentences_with_curly_bracket'] > 0) & (df_current['corpus'] == 'fineweb-2'))]

In [74]:
results.append(evaluate_filter_stage(
    df_base=df_baseline,
    df_prev=df_prev,
    df_current = df_current,
    stage_name = "2. No curly brackets opus"
))

results[-1]

Stage                      2. No curly brackets opus
Docs Kept                                     958458
Cum. Token Yield (%)                           98.57
Step Tokens Dropped (%)                         0.74
Weighted Rel NLL                             -1.1542
Step Gain (vs Prev)                           0.0019
Cum Gain (vs Base)                             0.008
Step Efficiency                              0.00262
dtype: object

In [78]:
df_prev = df_current.copy()

df_current = df_current[~((df_current['count_sentences_with_javascript'] > 0) & (df_current['corpus'] == 'opus-all-en') & (df_current['language_score'] < 0.95))]

df_current = df_current[~((df_current['count_sentences_with_javascript'] > 0) & (df_current['corpus'] == 'fineweb-2'))]

In [79]:
evaluate_filter_stage(
    df_base=df_baseline,
    df_prev=df_prev,
    df_current = df_current,
    stage_name = "3. Having JavaScript"
)

Stage                      3. Having JavaScript
Docs Kept                                958429
Cum. Token Yield (%)                      98.55
Step Tokens Dropped (%)                    0.02
Weighted Rel NLL                        -1.1542
Step Gain (vs Prev)                        -0.0
Cum Gain (vs Base)                       0.0079
Step Efficiency                        -0.00107
dtype: object

In [83]:
df_current = df_prev.copy()

df_current = df_current[~((df_current['mean_word_length'] < 3.0) & (df_current['corpus'] == 'opus-all-en'))]

In [84]:
evaluate_filter_stage(
    df_base=df_baseline,
    df_prev=df_prev,
    df_current = df_current,
    stage_name = "3. Low Mean Word Length Opus"
)

Stage                      3. Low Mean Word Length Opus
Docs Kept                                        954118
Cum. Token Yield (%)                              98.47
Step Tokens Dropped (%)                             0.1
Weighted Rel NLL                                -1.1563
Step Gain (vs Prev)                              0.0021
Cum Gain (vs Base)                                 0.01
Step Efficiency                                 0.02085
dtype: object

In [80]:
results

[Stage                      0. Baseline (Unfiltered)
 Docs Kept                                    959936
 Cum. Token Yield (%)                          100.0
 Step Tokens Dropped (%)                         0.0
 Weighted Rel NLL                            -1.1463
 Step Gain (vs Prev)                             0.0
 Cum Gain (vs Base)                              0.0
 Step Efficiency                                 0.0
 dtype: object,
 Stage                      1. Having sentences with legal phrases
 Docs Kept                                                  959074
 Cum. Token Yield (%)                                         99.3
 Step Tokens Dropped (%)                                       0.7
 Weighted Rel NLL                                          -1.1523
 Step Gain (vs Prev)                                         0.006
 Cum Gain (vs Base)                                          0.006
 Step Efficiency                                           0.00861
 dtype: object,
 Stage  

In [ ]:
df_all[(df_all['mean_word_length'] > 11.7) & (df_all['corpus'] == 'opus-all-en') & (df_all['count_sentences_with_low_guarani_proportion'] > 0)].to_json(str(OUTPUT_PATH/f'mwl_oae.jsonl'), orient='records', lines=True, force_ascii=False)

In [61]:
fig = px.histogram(df_all[df_all['count_sentences_ending_with_ellipsis'] > 0], x="avg_alphanumeric_characters_per_sentence",)
fig.show()

In [12]:
def extract_percentile(df:pd.DataFrame, column:str, percentile:float, modality:str, output_dir:Path):
    percentile_value = df[column].quantile(percentile)

    print(f"Percentile Value: {percentile_value}")

    if modality == ">":
        filtered_df = df[df[column] > percentile_value]
    elif modality == ">=":
        filtered_df = df[df[column] >= percentile_value]
    elif modality == "<":
        filtered_df = df[df[column] < percentile_value]
    elif modality == "<=":
        filtered_df = df[df[column] <= percentile_value]

    filtered_df.to_json(str(output_dir/f'{column}_{int(percentile*100)}.jsonl'), orient='records', lines=True, force_ascii=False)

In [13]:
extract_percentile(df_all, 'mean_word_length', 0.99, ">", OUTPUT_PATH)

Percentile Value: 11.166666666666666


In [95]:
baseline_eval_df = pd.read_json(str(OUTPUT_PATH/"eval"/"detailed_reports"/"baseline_qwen.jsonl"), lines=True, orient="records")
boilerplate_eval_df = pd.read_json(str(OUTPUT_PATH/"eval"/"detailed_reports"/"boilerplate_removal.jsonl"), lines=True, orient="records")

In [96]:
baseline_eval_df = baseline_eval_df[["question_id", "question", "ground_truth", "predicted", "is_correct"]].rename(
    columns={"predicted": "pred_baseline", "is_correct": "correct_baseline"}
)

boilerplate_eval_df = boilerplate_eval_df[["question_id", "predicted", "is_correct"]].rename(
    columns={"predicted": "pred_boilerplate", "is_correct": "correct_boilerplate"},
)

In [97]:
comparative = baseline_eval_df.merge(boilerplate_eval_df, on="question_id")

In [98]:
comparative['matching'] = (comparative['correct_baseline'] == comparative['correct_boilerplate'])

In [99]:
comparative['matching'].value_counts()

matching
True     399
False      1
Name: count, dtype: int64

In [100]:
labels = ["A", "B", "C", "D"]

In [45]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [101]:
fig_cm = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=(
            "Baseline Model",
            "Boilerplate Removal Filter",
        ),
        shared_yaxes=True,
)

In [102]:
confusion_baseline = pd.crosstab(
    comparative['ground_truth'],
    comparative['pred_baseline'],
    dropna=False
).reindex(index=labels, columns=labels, fill_value=0)

z_values_baseline = confusion_baseline.values
text_values_baseline = [[str(val) for val in row] for row in z_values_baseline]

confusion_boilerplate = pd.crosstab(
    comparative['ground_truth'],
    comparative['pred_boilerplate'],
    dropna=False
).reindex(index=labels, columns=labels, fill_value=0)

z_values_boilerplate = confusion_baseline.values
text_values_boilerplate = [[str(val) for val in row] for row in z_values_boilerplate]

In [103]:
baseline_cm = go.Heatmap(
            z=z_values_baseline,
            x=labels,
            y=labels,
            text=text_values_baseline,
            texttemplate="%{text}",
            textfont={"size": 14},
            colorscale="Blues",
            showscale=False,
)

fig_cm.add_trace(baseline_cm, row=1, col=1)

boilerplate_cm = go.Heatmap(
            z=z_values_boilerplate,
            x=labels,
            y=labels,
            text=text_values_boilerplate,
            texttemplate="%{text}",
            textfont={"size": 14},
            colorscale="Blues",
            showscale=False,
)

fig_cm.add_trace(boilerplate_cm, row=1, col=2)

In [78]:
def apply_boilerplate_filters(df:pd.DataFrame) -> pd.DataFrame:
    """Applies filters that are meant to erase boilerplate content"""

    #Remove documents with legal phrases
    df = df[~(df['count_sentences_with_legal_phrases'] > 0)]

    #Remove documents with curly brackets for the fineweb-2 and opus-all-en corpora
    df = df[~((df['count_sentences_with_curly_bracket'] > 0) & (df['corpus'] == 'opus-all-en') & (df['language_score'] < 0.9965))]
    df = df[~((df['count_sentences_with_curly_bracket'] > 0) & (df['corpus'] == 'fineweb-2'))]

    #Remove documents with javascript for the same corpora
    df = df[~((df['count_sentences_with_javascript'] > 0) & (df['corpus'] == 'opus-all-en') & (df['language_score'] < 0.95))]
    df = df[~((df['count_sentences_with_javascript'] > 0) & (df['corpus'] == 'fineweb-2'))]

    #Remove documents with low mean word length
    #TODO: also test with high word length
    corpus_to_remove = ['opus-all-en', 'opus', 'gua_spa', 'belele', 'FinePDF', 'josa']
    df = df[~((df['mean_word_length'] < 3.0) & (df['corpus'].isin(corpus_to_remove)))]

    df = df[~((df['ratio_symbols_to_words'] > 3) & (df['corpus'] == 'opus-all-en') & (df['language_score'] < 1))]
    
    df = df[~((df['count_sentences_ending_with_ellipsis'] > 0) & (df['corpus'] == 'opus-all-en') & (df['language_score'] < 1))]

    return df

In [79]:
df_boilerplate_filter = apply_boilerplate_filters(df_all.copy())

In [81]:
df_boilerplate_filter["num_words_split"].sum()

np.int64(37870349)

In [92]:
token_reduction = 38496060 - 37870349
trp = token_reduction / 38496060

In [93]:
token_reduction

625711

In [94]:
(1-trp)*100

98.37461028479278

In [87]:
doc_reduction = 959936 - len(df_boilerplate_filter)
drp = doc_reduction/959936

In [88]:
doc_reduction

9054

In [91]:
(1-drp)*100

99.05681212080806

In [ ]:
df_boilerplate_filter

In [52]:
df_boilerplate_filter[df_boilerplate_filter['mean_word_length'] > 11.17]

,id,text,count_lorem_ipsum_sentences,pii_spans,count_sentences_without_terminal_punctuation,coreguapa_perplexity,avg_sentence_length,language_score,corpus_file,avg_characters_per_sentence,...,has_pii,count_sentences_with_javascript,avg_numbers_per_sentence,language_script,url,avg_word_repetition_ratio_per_sentence,count_bad_words_occurrences,language,avg_words_per_sentence,tweets_perplexity
51591,smolsent__en_gn_17,Aguyjeme'ẽkuaaningo ohechauka jeporohayhukuaat...,0,[],0,37.966911,4.000000,0.999886,smolsent_en_gn.jsonl,103.000000,...,False,0,0.000000,Latn,unknown,0.0,0,grn,4.000000,121.374304
53172,guarania-scraped_476,Ndojeguerohorýiete irresponsabilidad oïva oñem...,0,[],1,50.874832,6.000000,0.997021,,73.000000,...,False,0,0.000000,Latn,https://www.abc.com.py/especiales/remiandu/abc...,0.0,0,grn,6.000000,247.922501
76217,jojajovai_310,Ambientalista ombokatupyry productor guaireño-...,0,[],1,1796.087014,3.000000,0.999968,data_jojajovai_all.csv,53.000000,...,False,0,0.000000,Latn,unknown,0.0,0,grn,3.000000,2262.745486
76223,jojajovai_316,Ndojeguerohorypái Comando-pe hikuái,0,[],1,66.398647,2.000000,0.998571,data_jojajovai_all.csv,35.000000,...,False,0,0.000000,Latn,unknown,0.0,0,grn,2.000000,142.308340
76570,jojajovai_663,Ñanemongyhyje.,0,[],0,315.912473,1.000000,1.000010,data_jojajovai_all.csv,14.000000,...,False,0,0.000000,Latn,unknown,0.0,0,grn,1.000000,1263.356016
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
948289,josa_3146,Ko'ápe nda'ijái py'ahatãite oporomongorapaséva...,0,[],2,87.405674,1.000000,0.999303,unbalanced_sa3_dev.txt,29.000000,...,False,0,0.000000,Latn,unknown,0.0,0,grn,1.000000,106.799622
948379,josa_3244,Eporanduromante eikuaata.,0,[],0,699.860567,2.000000,0.971141,unbalanced_sa3_dev.txt,25.000000,...,False,0,0.000000,Latn,unknown,0.0,0,grn,2.000000,109.518347
948473,josa_3347,[Jaikuaavemína – Conozcamos más] Pueblo indíge...,0,[],3,227.820451,2.666667,0.563925,unbalanced_sa3_dev.txt,52.333333,...,False,0,1.666667,Latn,unknown,0.0,0,grn,2.666667,120.364762
948516,josa_3395,¡AGUIJE! #SemanaDeLaPoesíaParaguaya https://t....,0,[],2,133.202852,1.000000,0.644246,unbalanced_sa3_dev.txt,37.000000,...,False,0,1.000000,Latn,unknown,0.0,0,grn,1.000000,76.262740


In [53]:
df_all.columns

Index(['id', 'text', 'count_lorem_ipsum_sentences', 'pii_spans',
       'count_sentences_without_terminal_punctuation', 'coreguapa_perplexity',
       'avg_sentence_length', 'language_score', 'corpus_file',
       'avg_characters_per_sentence', 'num_chars',
       'ratio_stopwords_to_non_stopwords', 'num_words_split',
       'count_sentences_with_low_guarani_proportion', 'pii_prop',
       'average_words_in_sentences_starting_with_capital',
       'avg_alphanumeric_characters_per_sentence', 'max_sentence_length',
       'ratio_symbols_to_words', 'mean_word_length', 'corpus',
       'min_sentence_length', 'count_sentences_ending_with_ellipsis',
       'language_identification_method', 'count_sentences_with_curly_bracket',
       'avg_uppercase_letters_per_sentence', 'source',
       'avg_character_repetition_ratio_per_sentence', 'duplicate',
       'num_words_punct_spacy', 'count_sentences_with_legal_phrases',
       'count_sentences_starting_with_bullet', 'language_score_source',
     

In [54]:
extract_percentile(df_all, "ratio_symbols_to_words",  0.99, ">", OUTPUT_PATH)

Percentile Value: 3.0


In [62]:
extract_percentile(df_all, "avg_alphanumeric_characters_per_sentence",  0.99, ">", OUTPUT_PATH)

Percentile Value: 311.0


In [ ]:
df_boilerplate_filter[df_boilerplate_filter['ratio_symbols_to_words'] > 3]

In [ ]:
common_count = len(df_all[df_all['avg_alphanumeric_characters_per_sentence'] > 300.0].merge(df_all[df_all['count_sentences_ending_with_ellipsis'] > 0], on="id", how="inner"))

In [66]:
df_all[df_all['avg_alphanumeric_characters_per_sentence'] > 200].merge(df_all[df_all['count_sentences_ending_with_ellipsis'] > 0], on="id", how="inner")

,id,text_x,count_lorem_ipsum_sentences_x,pii_spans_x,count_sentences_without_terminal_punctuation_x,coreguapa_perplexity_x,avg_sentence_length_x,language_score_x,corpus_file_x,avg_characters_per_sentence_x,...,has_pii_y,count_sentences_with_javascript_y,avg_numbers_per_sentence_y,language_script_y,url_y,avg_word_repetition_ratio_per_sentence_y,count_bad_words_occurrences_y,language_y,avg_words_per_sentence_y,tweets_perplexity_y
0,guarania-scraped_15907,MISIÓN y VISIÓN de la Facultad de Medicina (en...,0,[],0,35.597032,53.000000,0.998435,,357.000000,...,False,0,0.000000,Latn,https://facultad-de-medicina-blog.webnode.es/a...,0.150943,0,grn,53.000000,80.898844
1,guarania-scraped_27977,Ore Ru yvágape reiméva Tojejapo ne rembipota k...,0,[],0,50.043784,54.000000,0.994231,,401.000000,...,False,0,0.000000,Latn,http://guaranireko.blogspot.com/2015/06/padre-...,0.166667,0,grn,54.000000,127.746280
2,guarania-scraped_33306,Este espacio sirve para promocionar y difundir...,0,[],1,65.481615,66.500000,0.986863,,398.375000,...,False,0,0.625000,Latn,https://lenguaguarani.blogspot.com/2015/04/ten...,0.140840,0,grn,66.500000,78.841610
3,fineweb-2_1290,5 jasyapy 2013\n- 16:0016:00 5 jasyapy 2013 +4...,0,[],0,7.200051,702.000000,0.681256,gug_Latn_removed_train_000_00000.csv,5437.000000,...,False,0,872.000000,Latn,https://gn.wikipedia.org/wiki/Mba%27ech%C4%A9c...,0.706553,0,grn,702.000000,10.355924
4,fineweb-2_2798,Puruhára mba'emoĩmbyre\nIr a la navegación Ir ...,0,"[{'type': 'ip', 'start': 425, 'end': 438, 'tex...",2,22.782798,26.500000,0.913734,gug_Latn_removed_train_000_00000.csv,269.250000,...,True,0,36.000000,Latn,https://gn.wikipedia.org/wiki/Mba%27ech%C4%A9c...,0.149033,0,grn,26.500000,45.354944
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
141,opus-all-en_400694,"Ápe ha pépe ojehesy kavure, oñembojere mbeju, ...",0,[],0,23.022021,30.000000,0.995798,OPUS-NLLB_v1_mono_gn.txt,264.000000,...,False,0,0.000000,Latn,unknown,0.066667,0,grn,30.000000,77.767899
142,hplt-3_1366,KÁSO ÑEMOMBE'U: IRUNDY KUIMBA’E REMBIASAKUE Oh...,0,[],0,15.252163,91.750000,0.669567,7_1.jsonl.zst,648.000000,...,False,0,14.750000,Latn,http://guarani.over-blog.es/page/80,0.261846,0,grn,91.750000,30.710296
143,hplt-3_5594,Gran día de observación de aves en el Cono Sur...,0,[],1,35.877627,153.666667,0.738040,6_1.jsonl.zst,1005.000000,...,False,0,3.666667,Latn,http://www.guyra.org.py/index.php?lang=en,0.218903,0,grn,153.666667,91.075780
144,hplt-3_7814,Se viene el Gran día de observación de aves de...,0,[],0,47.100631,230.500000,0.686874,5_1.jsonl.zst,1573.500000,...,False,0,10.000000,Latn,http://www.guyra.org.py/index.php?lang=en,0.190265,0,grn,230.500000,97.988386


In [77]:
df_all[(df_all['count_sentences_ending_with_ellipsis'] > 0) & (df_all['corpus'] == 'opus-all-en') & (df_all['language_score'] > 1)].to_json(str(OUTPUT_PATH/f'ellipsis_oe_hls.jsonl'), orient='records', lines=True, force_ascii=False)